In [ ]:
from pathlib import Path

from covid.data import CovidDataset
from covid.config_loader import from_yaml
import pandas as pd

run_config = from_yaml(Path("../config/config.yaml"))
raw_df = pd.read_csv(".." / run_config.dataset.raw_path)
dataset = (
    CovidDataset.from_dataframe(
        raw_df,
        target=run_config.dataset.target_column,
        id_column=run_config.dataset.id_column
    )
    .without_sparse_columns(threshold=run_config.dataset.sparse_threshold)
    .as_categorical()
)

In [ ]:
X, y = dataset.split()

In [ ]:
y.value_counts()

In [ ]:
from umap import UMAP
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

umap_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("scaler", StandardScaler()),
    ("reducer", UMAP(n_components=2, n_neighbors=100, metric="manhattan"))
])
X_umap = umap_pipe.fit_transform(X)


In [ ]:
from sklearn.decomposition import PCA

pca_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("scaler", StandardScaler()),
    ("reducer", PCA(n_components=2))
])
X_pca = pca_pipe.fit_transform(X)

In [ ]:
from matplotlib import pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].scatter(X_umap[:, 0], X_umap[:, 1], c=y, cmap="viridis", alpha=0.7)
axes[0].set_title("UMAP Projection")
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap="viridis", alpha=0.7)
axes[1].set_title("PCA Projection")
plt.tight_layout()
plt.show()